In [1]:
import numpy as np
import pandas as pd
import kagglehub
from football import baseline, data_loader

/home/sabmel/vol3_final_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
path = kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016")
dl = data_loader.DataLoader(path=path)
dl.clean()

num_seasons = 8
num_games = 250

Simple Baseline

In [3]:
# doubles the first-half yardage and compares sign with actual
simple_correct = 0
total_count = 0

for i in range(num_seasons):
    for j in range(num_games):
        train_df, test_df = dl[i][j].train_test_split()
        # Get yardage arrays
        train_yards = train_df["team0_yards"].values.reshape(-1, 1)
        test_yards = test_df["team0_yards"].values.reshape(-1, 1)
        if len(train_yards) == 0 or len(test_yards) == 0:
            continue
        
        sb_result = baseline.simple_baseline(train_yards, test_yards)  # returns 0 or 1
        simple_correct += sb_result
        total_count += 1

if total_count > 0:
    simple_accuracy = simple_correct / total_count
    print(f"(A) Simple baseline accuracy: {simple_accuracy:.2%}")
else:
    print("(A) No valid data for simple baseline.")

(A) Simple baseline accuracy: 73.30%


Time-Series Baseline

In [4]:
# uses an AR model on the first half, sums predicted yardage for second half
# and checks sign correctness.
ts_correct = 0
total_count_ts = 0

for i in range(num_seasons):
    for j in range(num_games):
        train_df, test_df = dl[i][j].train_test_split()
        if len(train_df) == 0 or len(test_df) == 0:
            continue
        ts_result = baseline.time_series_baseline(train_df, test_df, lags=5)['sign_correct']  # 0 or 1
        ts_correct += ts_result
        total_count_ts += 1

if total_count_ts > 0:
    ts_accuracy = ts_correct / total_count_ts
    print(f"(B) Time-series baseline accuracy: {ts_accuracy:.2%}")
else:
    print("(B) No valid data for time-series baseline.")

(B) Time-series baseline accuracy: 48.35%


Logistic Regression at Game Level

In [5]:
# gather a single feature (1st half net yardage difference),
# and a binary outcome (did Team 0 outgain Team 1 in the 2nd half?).
features = []
labels = []

for i in range(num_seasons):
    for j in range(num_games):
        train_df, test_df = dl[i][j].train_test_split()
        if len(train_df) == 0 or len(test_df) == 0:
            continue
        X_val, y_val = baseline.get_game_level_data(train_df, test_df)
        features.append(X_val)
        labels.append(y_val)

if len(features) > 0:
    model = baseline.logistic_regression_game_level(features, labels)
else:
    print("(C) No valid data for logistic regression game-level baseline.")

Logistic Regression (full-game) Accuracy: 73.20%
